# Solar Irradiance Prediction (without Kathmandu)

This notebook predicts Solar Irradiance using XGBoost, Random Forest, KNN, LSTM, and ANN-MLP for non-Kathmandu regions (Nuwakot, Dhangadhi, Mustang, Taplejung).

Dataset1 (2014-2022) is used for training and test splitting (80/20).
Dataset2 (Jan-Sep 2023) is used for model evaluation.


In [19]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import os
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor

from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping, ModelCheckpoint
import tensorflow as tf

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Configuration
# Options: 'nuwakot_data.csv', 'dhangadhi_data.csv', 'mustang_data.csv', 'taplejung_data.csv'
DATA_FILE = "dhangadhi_data.csv"
REGION_NAME = DATA_FILE.split('_')[0].capitalize()

BASE_PATH = "/content/drive/Shared-with-me/Solar/Data_solar"
DATA_PATH = os.path.join(BASE_PATH, DATA_FILE)
MODEL_SAVE_DIR = "/content/drive/MyDrive/RESEARCH/Solar/Model_Works/model"

print(f"Selected Region: {REGION_NAME}")


Selected Region: Dhangadhi


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [16]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [17]:
# Create model directory if it doesn't exist
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)


In [18]:
# Load and inspect data
data = pd.read_csv(DATA_PATH)
print(f"Data shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
print(f"Date range: {data['YEAR'].min()}-{data['YEAR'].max()}")
data.head()


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/Sharedwithme/Solar/Data_solar/dhangadhi_data.csv'

In [ ]:
print(f"Checking contents of: {BASE_PATH}")
if os.path.exists(BASE_PATH):
    print(os.listdir(BASE_PATH))
else:
    print(f"The base path does not exist: {BASE_PATH}")

Checking contents of: /content/drive/MyDrive/RESEARCH/Solar/Data_solar
The base path does not exist: /content/drive/MyDrive/RESEARCH/Solar/Data_solar


In [ ]:
# Data preprocessing
# Convert date columns to numeric
date_cols = ['YEAR', 'MO', 'DY', 'HR']
for col in date_cols:
    data[col] = pd.to_numeric(data[col], errors='coerce')

# Drop rows with any NaN values
data = data.dropna()

# Drop problematic columns with many missing values (-999) and target-derived variables
cols_to_drop = [
    'ALLSKY_SRF_ALB', 'SZA', 'ALLSKY_KT',
    'CLRSKY_SFC_SW_DWN', 'ALLSKY_SFC_PAR_TOT', 'CLRSKY_SFC_PAR_TOT',
    'ALLSKY_SFC_UVA', 'ALLSKY_SFC_UVB', 'ALLSKY_SFC_UV_INDEX'
]
data = data.drop(cols_to_drop, axis=1, errors='ignore')

# Rename columns for clarity
rename_dict = {
    'YEAR': 'Year', 'MO': 'Month', 'DY': 'Day', 'HR': 'Hour',
    'ALLSKY_SFC_SW_DWN': 'Solar_Irradiance',
    'T2MDEW': 'DEW2M',
    'QV2M': 'SH2M',
    'PRECTOTCORR': 'Precipitation',
    'PS': 'SP'
}
data.rename(columns=rename_dict, inplace=True)

# Replace remaining -999 values with 0
data = data.replace(-999, 0)

print(f"Data shape after cleaning: {data.shape}")
data.head()


In [ ]:
# Exploratory Data Analysis (EDA)
plt.figure(figsize=(12, 6))
plt.hist(data['Solar_Irradiance'], bins=20, alpha=0.7, color='blue')
plt.xlabel('Solar Irradiance (Wh/m^2)')
plt.ylabel('Frequency')
plt.title(f'Distribution of Solar Irradiance - {REGION_NAME}')
plt.grid(True)
plt.show()


In [ ]:
# Correlation Heatmap
correlation_matrix = data.corr()
plt.figure(figsize=(12, 8))
sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm')
plt.title(f'Correlation Heatmap - {REGION_NAME}')
plt.show()


In [ ]:
# Split into Dataset1 (2014-2022) and Dataset2 (Jan-Sep 2023)
data1 = data[data['Year'] <= 2022].copy()
data2 = data[(data['Year'] == 2023) & (data['Month'] <= 9)].copy()

print(f"Dataset1 (2014-2022) shape: {data1.shape}")
print(f"Dataset2 (2023) shape: {data2.shape}")


In [ ]:
# Feature/Target Split for Dataset1
target_col = 'Solar_Irradiance'
X = data1.drop([target_col], axis=1)
y = data1[target_col]

# Train-Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")


In [ ]:
# Function to evaluate and print metrics
def evaluate_model(actual, predicted, model_name, dataset_name):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mbe = np.mean(actual - predicted)
    r2 = r2_score(actual, predicted)

    print(f"--- {model_name} on {dataset_name} ---")
    print(f"Mean Absolute Error (MAE): {mae:.4f}")
    print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
    print(f"Mean Bias Error (MBE): {mbe:.4f}")
    print(f"R-squared (R²): {r2:.4f}\n")
    return mae, rmse, mbe, r2

metrics_dict = {}


In [ ]:
# Prepare Dataset2 features and target
X_dataset2 = data2.drop([target_col], axis=1)
y_dataset2 = data2[target_col]


## 1. XGBoost Regressor


In [ ]:
xgb_model = XGBRegressor(random_state=42)
xgb_model.fit(X_train, y_train)

# Evaluate on Dataset1 Test set
xgb_pred_test = xgb_model.predict(X_test)
evaluate_model(y_test, xgb_pred_test, "XGBoost", "Dataset1 Test Set")

# Evaluate on Dataset2 (2023)
xgb_pred_dataset2 = xgb_model.predict(X_dataset2)
xgb_metrics = evaluate_model(y_dataset2, xgb_pred_dataset2, "XGBoost", "Dataset2 (2023)")
metrics_dict['XGBoost'] = xgb_metrics

# Save model
joblib.dump(xgb_model, os.path.join(MODEL_SAVE_DIR, f'xgboost_{REGION_NAME.lower()}.pkl'))


In [ ]:
# XGBoost Plots for Dataset2
plt.figure(figsize=(14, 6))
plt.plot(y_dataset2.values, label='Actual Solar Irradiance', color='blue', alpha=0.7)
plt.plot(xgb_pred_dataset2, label='Predicted Solar Irradiance', color='red', alpha=0.7)
plt.title(f'XGBoost - Actual vs Predicted (Dataset2) - {REGION_NAME}')
plt.xlabel('Data Point')
plt.ylabel('Solar Irradiance (Wh/m²)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 6))
plt.scatter(y_dataset2, xgb_pred_dataset2, alpha=0.5)
plt.plot([y_dataset2.min(), y_dataset2.max()], [y_dataset2.min(), y_dataset2.max()], 'r--', lw=2)
plt.xlabel('Actual Solar Irradiance')
plt.ylabel('Predicted Solar Irradiance')
plt.title('XGBoost - Scatter Plot (Dataset2)')
plt.grid(True, alpha=0.3)
plt.show()


## 2. Random Forest Regressor


In [ ]:
rf_model = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Evaluate on Dataset1 Test set
rf_pred_test = rf_model.predict(X_test)
evaluate_model(y_test, rf_pred_test, "Random Forest", "Dataset1 Test Set")

# Evaluate on Dataset2 (2023)
rf_pred_dataset2 = rf_model.predict(X_dataset2)
rf_metrics = evaluate_model(y_dataset2, rf_pred_dataset2, "Random Forest", "Dataset2 (2023)")
metrics_dict['Random Forest'] = rf_metrics

# Save model
joblib.dump(rf_model, os.path.join(MODEL_SAVE_DIR, f'randomforest_{REGION_NAME.lower()}.pkl'))


In [ ]:
# Random Forest Plots for Dataset2
plt.figure(figsize=(14, 6))
plt.plot(y_dataset2.values, label='Actual Solar Irradiance', color='blue', alpha=0.7)
plt.plot(rf_pred_dataset2, label='Predicted Solar Irradiance', color='red', alpha=0.7)
plt.title(f'Random Forest - Actual vs Predicted (Dataset2) - {REGION_NAME}')
plt.xlabel('Data Point')
plt.ylabel('Solar Irradiance (Wh/m²)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 3. K-Nearest Neighbors Regressor


In [ ]:
knn_model = KNeighborsRegressor()
knn_model.fit(X_train, y_train)

# Evaluate on Dataset1 Test set
knn_pred_test = knn_model.predict(X_test)
evaluate_model(y_test, knn_pred_test, "KNN", "Dataset1 Test Set")

# Evaluate on Dataset2 (2023)
knn_pred_dataset2 = knn_model.predict(X_dataset2)
knn_metrics = evaluate_model(y_dataset2, knn_pred_dataset2, "KNN", "Dataset2 (2023)")
metrics_dict['KNN'] = knn_metrics

# Save model
joblib.dump(knn_model, os.path.join(MODEL_SAVE_DIR, f'knn_{REGION_NAME.lower()}.pkl'))


In [ ]:
# KNN Plots for Dataset2
plt.figure(figsize=(14, 6))
plt.plot(y_dataset2.values, label='Actual Solar Irradiance', color='blue', alpha=0.7)
plt.plot(knn_pred_dataset2, label='Predicted Solar Irradiance', color='red', alpha=0.7)
plt.title(f'KNN - Actual vs Predicted (Dataset2) - {REGION_NAME}')
plt.xlabel('Data Point')
plt.ylabel('Solar Irradiance (Wh/m²)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 4. LSTM Neural Network


In [ ]:
# Standardize the data for deep learning models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_dataset2_scaled = scaler.transform(X_dataset2)

# Save scaler
joblib.dump(scaler, os.path.join(MODEL_SAVE_DIR, f'scaler_{REGION_NAME.lower()}.pkl'))

# Reshape for LSTM [samples, time steps, features]
X_train_lstm = X_train_scaled.reshape(X_train_scaled.shape[0], 1, X_train_scaled.shape[1])
X_test_lstm = X_test_scaled.reshape(X_test_scaled.shape[0], 1, X_test_scaled.shape[1])
X_dataset2_lstm = X_dataset2_scaled.reshape(X_dataset2_scaled.shape[0], 1, X_dataset2_scaled.shape[1])


In [ ]:
# Build LSTM Model
lstm_model = Sequential([
    LSTM(64, activation='relu', return_sequences=True, input_shape=(1, X_train_scaled.shape[1])),
    Dropout(0.2),
    LSTM(32, activation='relu'),
    Dense(1)
])

lstm_model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error', metrics=['mae'])

early_stopping = EarlyStopping(
    monitor='val_loss',
    min_delta=0.001,
    patience=10,
    restore_best_weights=True
)

lstm_history = lstm_model.fit(
    X_train_lstm, y_train,
    batch_size=32,
    epochs=100,
    validation_data=(X_test_lstm, y_test),
    callbacks=[early_stopping],
    verbose=1
)


In [ ]:
# Plot LSTM Training History
plt.figure(figsize=(10, 4))
plt.plot(lstm_history.history['loss'], label='Train Loss')
plt.plot(lstm_history.history['val_loss'], label='Validation Loss')
plt.title('LSTM Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Evaluate on Dataset1 Test set
lstm_pred_test = lstm_model.predict(X_test_lstm, verbose=0).flatten()
evaluate_model(y_test, lstm_pred_test, "LSTM", "Dataset1 Test Set")

# Evaluate on Dataset2 (2023)
lstm_pred_dataset2 = lstm_model.predict(X_dataset2_lstm, verbose=0).flatten()
lstm_metrics = evaluate_model(y_dataset2, lstm_pred_dataset2, "LSTM", "Dataset2 (2023)")
metrics_dict['LSTM'] = lstm_metrics

# Save model
lstm_model.save(os.path.join(MODEL_SAVE_DIR, f'lstm_{REGION_NAME.lower()}.h5'))


In [ ]:
# LSTM Plots for Dataset2
plt.figure(figsize=(14, 6))
plt.plot(y_dataset2.values, label='Actual Solar Irradiance', color='blue', alpha=0.7)
plt.plot(lstm_pred_dataset2, label='Predicted Solar Irradiance', color='red', alpha=0.7)
plt.title(f'LSTM - Actual vs Predicted (Dataset2) - {REGION_NAME}')
plt.xlabel('Data Point')
plt.ylabel('Solar Irradiance (Wh/m²)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## 5. Artificial Neural Network (ANN-MLP)


In [ ]:
# Build ANN-MLP Model
ann_model = Sequential([
    Dense(16, activation='relu', input_dim=X_train_scaled.shape[1]),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')
])

ann_model.compile(optimizer='adam', loss='mean_squared_error', metrics=['mae'])

early_stopping_ann = EarlyStopping(
    monitor='val_loss',
    min_delta=0.00001,
    patience=15,
    restore_best_weights=True
)

ann_history = ann_model.fit(
    X_train_scaled, y_train,
    batch_size=32,
    epochs=200,
    validation_data=(X_test_scaled, y_test),
    callbacks=[early_stopping_ann],
    verbose=1
)


In [ ]:
# Plot ANN Training History
plt.figure(figsize=(10, 4))
plt.plot(ann_history.history['loss'], label='Train Loss')
plt.plot(ann_history.history['val_loss'], label='Validation Loss')
plt.title('ANN-MLP Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


In [ ]:
# Evaluate on Dataset1 Test set
ann_pred_test = ann_model.predict(X_test_scaled, verbose=0).flatten()
evaluate_model(y_test, ann_pred_test, "ANN-MLP", "Dataset1 Test Set")

# Evaluate on Dataset2 (2023)
ann_pred_dataset2 = ann_model.predict(X_dataset2_scaled, verbose=0).flatten()
ann_metrics = evaluate_model(y_dataset2, ann_pred_dataset2, "ANN-MLP", "Dataset2 (2023)")
metrics_dict['ANN-MLP'] = ann_metrics

# Save model
ann_model.save(os.path.join(MODEL_SAVE_DIR, f'ann_{REGION_NAME.lower()}.h5'))


In [ ]:
# ANN Plots for Dataset2
plt.figure(figsize=(14, 6))
plt.plot(y_dataset2.values, label='Actual Solar Irradiance', color='blue', alpha=0.7)
plt.plot(ann_pred_dataset2, label='Predicted Solar Irradiance', color='red', alpha=0.7)
plt.title(f'ANN-MLP - Actual vs Predicted (Dataset2) - {REGION_NAME}')
plt.xlabel('Data Point')
plt.ylabel('Solar Irradiance (Wh/m²)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Summary of Model Performances on Dataset2 (2023)


In [ ]:
# Create a summary DataFrame
metrics_df = pd.DataFrame.from_dict(
    metrics_dict,
    orient='index',
    columns=['MAE', 'RMSE', 'MBE', 'R²']
)

print(f"--- Performance Comparison on Dataset2 ({REGION_NAME}) ---")
display(metrics_df.round(4))
